> **Versión GitHub reproducible.** Esta copia no modifica el notebook original de Google Drive. Las rutas personales de Drive se han sustituido por rutas del repositorio. Los datos brutos de gran tamaño no incluidos en GitHub deben descargarse según `docs/DATA_SOURCES.md`.


In [ ]:
# ============================================================
# CONFIGURACIÓN DE RUTAS - VERSIÓN GITHUB / GOOGLE COLAB
# ============================================================
# Clone primero el repositorio con:
# !git clone https://github.com/ritaridh/tfm-crop-yield-prediction.git /content/TFM_Rendimiento_cultivos
# %cd /content/TFM_Rendimiento_cultivos

from pathlib import Path
import os

ROOT = Path("/content/TFM_Rendimiento_cultivos")
if not ROOT.exists():
    raise FileNotFoundError(
        "No se encuentra el repositorio. Ejecute primero el bloque de clonación indicado en el README."
    )

RUTA_TFM = str(ROOT)
RUTA_BASE = str(ROOT)
BASE = str(ROOT)
BASE_DIR = ROOT
RUTA_ORIGINALES = str(ROOT / "01_Datos_originales")
RUTA_EUROCROPS = str(ROOT / "01_Datos_originales" / "EuroCrops")
RUTA_MAPA = str(ROOT / "01_Datos_originales" / "MAPA")
RUTA_SENTINEL = str(ROOT / "01_Datos_originales" / "Sentinel2")
RUTA_PROCESADOS = str(ROOT / "02_Datos_procesados")
RUTA_RESULTADOS = str(ROOT / "05_Resultados")
PROC = RUTA_PROCESADOS
RESULTADOS = RUTA_RESULTADOS
Path(RUTA_PROCESADOS).mkdir(parents=True, exist_ok=True)
Path(RUTA_RESULTADOS).mkdir(parents=True, exist_ok=True)
print("Repositorio:", ROOT)


In [ ]:
# ============================================================
# PASO 1 - CONEXIÓN CON GOOGLE DRIVE Y VERIFICACIÓN
#          DEL DATASET MAESTRO DEFINITIVO
# ============================================================

import pandas as pd
import os

# ------------------------------------------------------------
# 1. MONTAR GOOGLE DRIVE
# ------------------------------------------------------------



# ------------------------------------------------------------
# 2. DEFINIR RUTAS DEL TFM
# ------------------------------------------------------------

ROOT = "/content/TFM_Rendimiento_cultivos"

PROC = f"{ROOT}/02_Datos_procesados"

RESULTADOS = f"{ROOT}/05_Resultados"

DL_DIR = f"{RESULTADOS}/Deep_Learning_Comparacion"

os.makedirs(
    DL_DIR,
    exist_ok=True
)


# ------------------------------------------------------------
# 3. DATASET MAESTRO DEFINITIVO
# ------------------------------------------------------------

F_DATASET = (
    f"{PROC}/"
    "dataset_maestro_TFM_2017_2023_DEFINITIVO.csv"
)

print("=" * 80)
print("PASO 1 - VERIFICACIÓN DEL ENTORNO")
print("=" * 80)

print("\nRuta raíz:")
print(ROOT)

print("\nDataset:")
print(F_DATASET)

print("\nDataset existe:")
print(os.path.exists(F_DATASET))


# ------------------------------------------------------------
# 4. CARGAR DATASET
# ------------------------------------------------------------

assert os.path.exists(F_DATASET), (
    "ERROR: no se encuentra el dataset maestro definitivo."
)

df = pd.read_csv(F_DATASET)


# ------------------------------------------------------------
# 5. COMPROBACIONES ESENCIALES
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("DATASET MAESTRO")
print("=" * 80)

print("\nDimensiones:")
print(df.shape)

print("\nAños:")
print(sorted(df["ano"].unique()))

print("\nNúmero de provincias:")
print(df["provincia_estandar"].nunique())

print("\nCultivos:")
print(df["cultivo"].value_counts())

print("\nTarget NaN:")
print(df["rendimiento_kg_ha"].isna().sum())

print("\nDuplicados de filas completas:")
print(df.duplicated().sum())


# ------------------------------------------------------------
# 6. DISTRIBUCIÓN TEMPORAL
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("OBSERVACIONES POR AÑO")
print("=" * 80)

print(
    df["ano"]
    .value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# 7. SEPARACIÓN QUE UTILIZAREMOS
# ------------------------------------------------------------

n_desarrollo = (
    df["ano"]
    .between(2017, 2022)
    .sum()
)

n_test = (
    (df["ano"] == 2023)
    .sum()
)

print("\n" + "=" * 80)
print("DISEÑO EXPERIMENTAL")
print("=" * 80)

print(
    "Desarrollo 2017-2022:",
    n_desarrollo
)

print(
    "Test independiente 2023:",
    n_test
)

print(
    "\n2023 permanecerá bloqueado durante "
    "la comparación y selección de arquitecturas."
)

print("\nPASO 1 FINALIZADO")

In [ ]:
# ============================================================
# PASO 2A - IDENTIFICAR COLUMNAS REALES DEL DATASET
# ============================================================

print("=" * 80)
print("PASO 2A - COLUMNAS DEL DATASET MAESTRO DEFINITIVO")
print("=" * 80)

print("\nNúmero total de columnas:", len(df.columns))

for i, col in enumerate(df.columns, start=1):
    print(f"{i:02d}. {col}")

print("\nPASO 2A FINALIZADO")

In [ ]:
# ============================================================
# PASO 2B - CONSTRUCCIÓN DEFINITIVA DE INPUTS COMUNES
#           PARA LAS 4 ARQUITECTURAS DEEP LEARNING
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 1. ERA5-LAND TEMPORAL
#    5 periodos × 4 variables
# ============================================================

periodos_era5 = [
    "ene_feb",
    "mar_abr",
    "may_jun",
    "jul_ago",
    "sep_dic"
]

variables_era5 = [
    "temp_media_c",
    "temp_rocio_c",
    "precipitacion_mm",
    "radiacion_mj_m2"
]

cols_era5 = [
    f"{variable}_{periodo}"
    for periodo in periodos_era5
    for variable in variables_era5
]

faltantes_era5 = [
    c for c in cols_era5
    if c not in df.columns
]

assert len(faltantes_era5) == 0, (
    f"Faltan columnas ERA5: {faltantes_era5}"
)

X_era5_seq = np.stack(
    [
        df[
            [
                f"{variable}_{periodo}"
                for variable in variables_era5
            ]
        ].to_numpy(dtype=np.float32)

        for periodo in periodos_era5
    ],
    axis=1
)


# ============================================================
# 2. SENTINEL-2 TEMPORAL
#    5 periodos × 3 índices
# ============================================================

periodos_sentinel = [
    "jan_feb",
    "mar_apr",
    "may_jun",
    "jul",
    "nov_dec"
]

indices_sentinel = [
    "NDVI",
    "EVI",
    "NDMI"
]

cols_sentinel = [
    f"{indice}_{periodo}"
    for periodo in periodos_sentinel
    for indice in indices_sentinel
]

faltantes_sentinel = [
    c for c in cols_sentinel
    if c not in df.columns
]

assert len(faltantes_sentinel) == 0, (
    f"Faltan columnas Sentinel-2: {faltantes_sentinel}"
)

X_sentinel_seq = np.stack(
    [
        df[
            [
                f"{indice}_{periodo}"
                for indice in indices_sentinel
            ]
        ].to_numpy(dtype=np.float32)

        for periodo in periodos_sentinel
    ],
    axis=1
)


# ============================================================
# 3. VARIABLES ESTÁTICAS / AGREGADAS
#    5 SoilGrids + 3 EuroCrops + 4 ERA5 campaña = 12
# ============================================================

cols_static = [

    # SoilGrids: 5
    "clay_pct_0_30cm",
    "sand_pct_0_30cm",
    "soc_g_kg_0_30cm",
    "ph_0_30cm",
    "bdod_g_cm3_0_30cm",

    # EuroCrops: 3
    "n_recintos",
    "superficie_eurocrops_ha",
    "superficie_media_recinto_ha",

    # ERA5-Land agregadas de campaña: 4
    "temp_media_c_campana",
    "temp_rocio_c_campana",
    "precipitacion_mm_campana",
    "radiacion_mj_m2_campana"
]

faltantes_static = [
    c for c in cols_static
    if c not in df.columns
]

assert len(faltantes_static) == 0, (
    f"Faltan variables estáticas/agregadas: {faltantes_static}"
)

X_static = df[
    cols_static
].to_numpy(dtype=np.float32)


# ============================================================
# 4. CULTIVO
# ============================================================

mapa_cultivo = {
    "barley": 0.0,
    "common_soft_wheat": 1.0
}

cultivos_no_reconocidos = (
    set(df["cultivo"].unique())
    - set(mapa_cultivo.keys())
)

assert len(cultivos_no_reconocidos) == 0, (
    f"Cultivos no reconocidos: {cultivos_no_reconocidos}"
)

X_crop = (
    df["cultivo"]
    .map(mapa_cultivo)
    .to_numpy(dtype=np.float32)
    .reshape(-1, 1)
)


# ============================================================
# 5. TARGET
# ============================================================

y = (
    df["rendimiento_kg_ha"]
    .to_numpy(dtype=np.float32)
)


# ============================================================
# 6. AÑO
# ============================================================

years = (
    df["ano"]
    .to_numpy()
)


# ============================================================
# 7. CONTROLES
# ============================================================

assert X_era5_seq.shape == (len(df), 5, 4)
assert X_sentinel_seq.shape == (len(df), 5, 3)
assert X_static.shape == (len(df), 12)
assert X_crop.shape == (len(df), 1)
assert y.shape == (len(df),)


print("=" * 80)
print("PASO 2B - INPUTS COMUNES DEFINITIVOS")
print("=" * 80)

print("\nDIMENSIONES")

print("ERA5-Land       :", X_era5_seq.shape)
print("Sentinel-2      :", X_sentinel_seq.shape)
print("Static/agregadas:", X_static.shape)
print("Cultivo         :", X_crop.shape)
print("Target          :", y.shape)


print("\nNaN antes del preprocesamiento")

print("ERA5-Land :", np.isnan(X_era5_seq).sum())
print("Sentinel-2:", np.isnan(X_sentinel_seq).sum())
print("Static     :", np.isnan(X_static).sum())
print("Cultivo    :", np.isnan(X_crop).sum())
print("Target     :", np.isnan(y).sum())


print("\nVARIABLES ESTÁTICAS / AGREGADAS")

for i, col in enumerate(cols_static, start=1):
    print(f"{i:02d}. {col}")


print("\nESTRUCTURA DE ENTRADA COMÚN")

print("ERA5-Land       : 5 periodos × 4 variables = 20")
print("Sentinel-2      : 5 periodos × 3 índices   = 15")
print("Static/agregadas: 12 variables")
print("Cultivo         : 1 variable")
print("----------------------------------------------")
print("Información predictora total equivalente: 48 entradas")
print("47 predictores numéricos + indicador de cultivo")


print("\nARQUITECTURAS A COMPARAR")
print("1. LSTM multirrama")
print("2. GRU multirrama")
print("3. CNN-1D multirrama")
print("4. ConvLSTM1D multirrama")


print("\nCONTROL METODOLÓGICO")
print("Dataset definitivo:", df.shape)
print("Desarrollo 2017-2022:", np.sum(years <= 2022))
print("Test independiente 2023:", np.sum(years == 2023))
print("2023 utilizado en este paso para entrenamiento: NO")


print("\nPASO 2B FINALIZADO")

In [ ]:
# ============================================================
# PASO 3 - PREPROCESAMIENTO COMÚN POR FOLD TEMPORAL
#          PARA LAS 4 ARQUITECTURAS DEEP LEARNING
# ============================================================

import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler


# ============================================================
# 1. FUNCIÓN COMÚN
# ============================================================

def preparar_fold_dl_comun(ano_validacion):
    """
    Prepara un fold temporal para Deep Learning.

    Train:
        todos los años anteriores al año de validación.

    Validación:
        únicamente el año indicado.

    IMPORTANTE:
    - Imputadores ajustados solo con TRAIN.
    - Escaladores ajustados solo con TRAIN.
    - 2023 nunca interviene en estos folds.
    """

    # --------------------------------------------------------
    # MÁSCARAS
    # --------------------------------------------------------

    mask_train = years < ano_validacion
    mask_val = years == ano_validacion


    # ========================================================
    # ERA5-LAND
    # ========================================================

    era5_train = (
        X_era5_seq[mask_train]
        .astype(np.float32)
    )

    era5_val = (
        X_era5_seq[mask_val]
        .astype(np.float32)
    )

    # Transformación temporal a 2D
    # para ajustar StandardScaler
    era5_train_2d = (
        era5_train.reshape(-1, 4)
    )

    era5_val_2d = (
        era5_val.reshape(-1, 4)
    )

    scaler_era5 = StandardScaler()

    era5_train_scaled = (
        scaler_era5
        .fit_transform(
            era5_train_2d
        )
    )

    era5_val_scaled = (
        scaler_era5
        .transform(
            era5_val_2d
        )
    )

    era5_train_scaled = (
        era5_train_scaled
        .reshape(era5_train.shape)
        .astype(np.float32)
    )

    era5_val_scaled = (
        era5_val_scaled
        .reshape(era5_val.shape)
        .astype(np.float32)
    )


    # ========================================================
    # SENTINEL-2
    # ========================================================

    sentinel_train = (
        X_sentinel_seq[mask_train]
        .astype(np.float32)
    )

    sentinel_val = (
        X_sentinel_seq[mask_val]
        .astype(np.float32)
    )

    sentinel_train_2d = (
        sentinel_train.reshape(-1, 3)
    )

    sentinel_val_2d = (
        sentinel_val.reshape(-1, 3)
    )


    # --------------------------------------------------------
    # Imputación
    # --------------------------------------------------------

    imputer_sentinel = SimpleImputer(
        strategy="median"
    )

    sentinel_train_imp = (
        imputer_sentinel
        .fit_transform(
            sentinel_train_2d
        )
    )

    sentinel_val_imp = (
        imputer_sentinel
        .transform(
            sentinel_val_2d
        )
    )


    # --------------------------------------------------------
    # Escalado
    # --------------------------------------------------------

    scaler_sentinel = StandardScaler()

    sentinel_train_scaled = (
        scaler_sentinel
        .fit_transform(
            sentinel_train_imp
        )
    )

    sentinel_val_scaled = (
        scaler_sentinel
        .transform(
            sentinel_val_imp
        )
    )

    sentinel_train_scaled = (
        sentinel_train_scaled
        .reshape(sentinel_train.shape)
        .astype(np.float32)
    )

    sentinel_val_scaled = (
        sentinel_val_scaled
        .reshape(sentinel_val.shape)
        .astype(np.float32)
    )


    # ========================================================
    # VARIABLES ESTÁTICAS / AGREGADAS
    # ========================================================

    static_train = (
        X_static[mask_train]
        .astype(np.float32)
    )

    static_val = (
        X_static[mask_val]
        .astype(np.float32)
    )


    # --------------------------------------------------------
    # Imputación
    # --------------------------------------------------------

    imputer_static = SimpleImputer(
        strategy="median"
    )

    static_train_imp = (
        imputer_static
        .fit_transform(
            static_train
        )
    )

    static_val_imp = (
        imputer_static
        .transform(
            static_val
        )
    )


    # --------------------------------------------------------
    # Escalado
    # --------------------------------------------------------

    scaler_static = StandardScaler()

    static_train_scaled = (
        scaler_static
        .fit_transform(
            static_train_imp
        )
        .astype(np.float32)
    )

    static_val_scaled = (
        scaler_static
        .transform(
            static_val_imp
        )
        .astype(np.float32)
    )


    # ========================================================
    # CULTIVO
    # ========================================================

    crop_train = (
        X_crop[mask_train]
        .astype(np.float32)
    )

    crop_val = (
        X_crop[mask_val]
        .astype(np.float32)
    )


    # ========================================================
    # TARGET
    # ========================================================

    y_train = (
        y[mask_train]
        .astype(np.float32)
    )

    y_val = (
        y[mask_val]
        .astype(np.float32)
    )


    # ========================================================
    # DEVOLVER RESULTADOS
    # ========================================================

    return {
        "X_train": [
            era5_train_scaled,
            sentinel_train_scaled,
            static_train_scaled,
            crop_train
        ],

        "y_train": y_train,

        "X_val": [
            era5_val_scaled,
            sentinel_val_scaled,
            static_val_scaled,
            crop_val
        ],

        "y_val": y_val,

        "n_train": int(mask_train.sum()),

        "n_val": int(mask_val.sum()),

        "train_anos": sorted(
            np.unique(
                years[mask_train]
            ).tolist()
        ),

        "val_ano": int(ano_validacion),

        # Guardamos también los preprocesadores
        # por trazabilidad y reproducibilidad
        "preprocesadores": {
            "scaler_era5":
                scaler_era5,

            "imputer_sentinel":
                imputer_sentinel,

            "scaler_sentinel":
                scaler_sentinel,

            "imputer_static":
                imputer_static,

            "scaler_static":
                scaler_static
        }
    }


# ============================================================
# 2. PRUEBA NECESARIA CON LOS 3 FOLDS
# ============================================================

print("=" * 80)
print("PASO 3 - PREPROCESAMIENTO TEMPORAL COMÚN")
print("=" * 80)

for ano_val in [
    2020,
    2021,
    2022
]:

    fold = preparar_fold_dl_comun(
        ano_val
    )

    print(
        f"\nValidación {ano_val}"
    )

    print(
        "Train:",
        fold["train_anos"]
    )

    print(
        "n_train:",
        fold["n_train"]
    )

    print(
        "n_val:",
        fold["n_val"]
    )

    print(
        "ERA5:",
        fold["X_train"][0].shape,
        "->",
        fold["X_val"][0].shape
    )

    print(
        "Sentinel:",
        fold["X_train"][1].shape,
        "->",
        fold["X_val"][1].shape
    )

    print(
        "Static:",
        fold["X_train"][2].shape,
        "->",
        fold["X_val"][2].shape
    )

    print(
        "Cultivo:",
        fold["X_train"][3].shape,
        "->",
        fold["X_val"][3].shape
    )


print("\nCONTROL METODOLÓGICO")

print(
    "Imputación ajustada dentro de cada fold: SÍ"
)

print(
    "Escalado ajustado dentro de cada fold: SÍ"
)

print(
    "2023 utilizado: NO"
)

print(
    "\nEste mismo preprocesamiento se utilizará "
    "para las cuatro arquitecturas."
)

print("\nPASO 3 FINALIZADO")

In [ ]:
# ============================================================
# PASO 4 - DEFINICIÓN DE LAS 4 ARQUITECTURAS
#          DEEP LEARNING MULTIRRAMA
# ============================================================

import tensorflow as tf

from tensorflow.keras import Model
from tensorflow.keras.layers import (
    Input,
    LSTM,
    GRU,
    Conv1D,
    ConvLSTM1D,
    GlobalAveragePooling1D,
    GlobalAveragePooling2D,
    Dense,
    Dropout,
    Concatenate,
    BatchNormalization,
    Reshape
)

from tensorflow.keras.regularizers import l2


# ============================================================
# CONFIGURACIÓN COMÚN
# ============================================================

SEED_DL = 42

DROPOUT_RATE = 0.25
L2_REG = 1e-4
LEARNING_RATE = 1e-3


# ============================================================
# FUNCIÓN AUXILIAR:
# RAMAS ESTÁTICA Y CULTIVO + FUSIÓN FINAL
# ============================================================

def construir_salida_comun(
    x_era5,
    x_sentinel,
    input_static,
    input_crop
):

    # --------------------------------------------------------
    # Rama estática
    # --------------------------------------------------------

    x_static = Dense(
        16,
        activation="relu",
        kernel_regularizer=l2(L2_REG),
        name="Dense_Static"
    )(input_static)

    x_static = BatchNormalization(
        name="BatchNorm_Static"
    )(x_static)

    x_static = Dropout(
        DROPOUT_RATE,
        name="Dropout_Static"
    )(x_static)


    # --------------------------------------------------------
    # Rama cultivo
    # --------------------------------------------------------

    x_crop = Dense(
        4,
        activation="relu",
        name="Dense_Crop"
    )(input_crop)


    # --------------------------------------------------------
    # Fusión
    # --------------------------------------------------------

    fusion = Concatenate(
        name="Fusion_multifuente"
    )([
        x_era5,
        x_sentinel,
        x_static,
        x_crop
    ])


    x = Dense(
        24,
        activation="relu",
        kernel_regularizer=l2(L2_REG),
        name="Dense_Fusion_1"
    )(fusion)

    x = BatchNormalization(
        name="BatchNorm_Fusion"
    )(x)

    x = Dropout(
        DROPOUT_RATE,
        name="Dropout_Fusion"
    )(x)


    x = Dense(
        8,
        activation="relu",
        kernel_regularizer=l2(L2_REG),
        name="Dense_Fusion_2"
    )(x)


    output = Dense(
        1,
        activation="linear",
        name="Rendimiento_kg_ha"
    )(x)

    return output


# ============================================================
# 1. LSTM MULTIRRAMA
# ============================================================

def crear_modelo_lstm():

    input_era5 = Input(
        shape=(5, 4),
        name="ERA5_Land"
    )

    input_sentinel = Input(
        shape=(5, 3),
        name="Sentinel_2"
    )

    input_static = Input(
        shape=(12,),
        name="Variables_estaticas"
    )

    input_crop = Input(
        shape=(1,),
        name="Cultivo"
    )


    x_era5 = LSTM(
        16,
        kernel_regularizer=l2(L2_REG),
        recurrent_regularizer=l2(L2_REG),
        name="LSTM_ERA5"
    )(input_era5)

    x_era5 = Dropout(
        DROPOUT_RATE,
        name="Dropout_ERA5"
    )(x_era5)


    x_sentinel = LSTM(
        12,
        kernel_regularizer=l2(L2_REG),
        recurrent_regularizer=l2(L2_REG),
        name="LSTM_Sentinel"
    )(input_sentinel)

    x_sentinel = Dropout(
        DROPOUT_RATE,
        name="Dropout_Sentinel"
    )(x_sentinel)


    output = construir_salida_comun(
        x_era5,
        x_sentinel,
        input_static,
        input_crop
    )


    model = Model(
        inputs=[
            input_era5,
            input_sentinel,
            input_static,
            input_crop
        ],
        outputs=output,
        name="LSTM_multirrama"
    )

    return compilar_modelo(model)


# ============================================================
# 2. GRU MULTIRRAMA
# ============================================================

def crear_modelo_gru():

    input_era5 = Input(
        shape=(5, 4),
        name="ERA5_Land"
    )

    input_sentinel = Input(
        shape=(5, 3),
        name="Sentinel_2"
    )

    input_static = Input(
        shape=(12,),
        name="Variables_estaticas"
    )

    input_crop = Input(
        shape=(1,),
        name="Cultivo"
    )


    x_era5 = GRU(
        16,
        kernel_regularizer=l2(L2_REG),
        recurrent_regularizer=l2(L2_REG),
        name="GRU_ERA5"
    )(input_era5)

    x_era5 = Dropout(
        DROPOUT_RATE,
        name="Dropout_ERA5"
    )(x_era5)


    x_sentinel = GRU(
        12,
        kernel_regularizer=l2(L2_REG),
        recurrent_regularizer=l2(L2_REG),
        name="GRU_Sentinel"
    )(input_sentinel)

    x_sentinel = Dropout(
        DROPOUT_RATE,
        name="Dropout_Sentinel"
    )(x_sentinel)


    output = construir_salida_comun(
        x_era5,
        x_sentinel,
        input_static,
        input_crop
    )


    model = Model(
        inputs=[
            input_era5,
            input_sentinel,
            input_static,
            input_crop
        ],
        outputs=output,
        name="GRU_multirrama"
    )

    return compilar_modelo(model)


# ============================================================
# 3. CNN-1D MULTIRRAMA
# ============================================================

def crear_modelo_cnn1d():

    input_era5 = Input(
        shape=(5, 4),
        name="ERA5_Land"
    )

    input_sentinel = Input(
        shape=(5, 3),
        name="Sentinel_2"
    )

    input_static = Input(
        shape=(12,),
        name="Variables_estaticas"
    )

    input_crop = Input(
        shape=(1,),
        name="Cultivo"
    )


    x_era5 = Conv1D(
        filters=16,
        kernel_size=2,
        padding="same",
        activation="relu",
        kernel_regularizer=l2(L2_REG),
        name="Conv1D_ERA5"
    )(input_era5)

    x_era5 = GlobalAveragePooling1D(
        name="GAP_ERA5"
    )(x_era5)

    x_era5 = Dropout(
        DROPOUT_RATE,
        name="Dropout_ERA5"
    )(x_era5)


    x_sentinel = Conv1D(
        filters=12,
        kernel_size=2,
        padding="same",
        activation="relu",
        kernel_regularizer=l2(L2_REG),
        name="Conv1D_Sentinel"
    )(input_sentinel)

    x_sentinel = GlobalAveragePooling1D(
        name="GAP_Sentinel"
    )(x_sentinel)

    x_sentinel = Dropout(
        DROPOUT_RATE,
        name="Dropout_Sentinel"
    )(x_sentinel)


    output = construir_salida_comun(
        x_era5,
        x_sentinel,
        input_static,
        input_crop
    )


    model = Model(
        inputs=[
            input_era5,
            input_sentinel,
            input_static,
            input_crop
        ],
        outputs=output,
        name="CNN1D_multirrama"
    )

    return compilar_modelo(model)


# ============================================================
# 4. ConvLSTM1D MULTIRRAMA
# ============================================================

def crear_modelo_convlstm1d():

    input_era5 = Input(
        shape=(5, 4),
        name="ERA5_Land"
    )

    input_sentinel = Input(
        shape=(5, 3),
        name="Sentinel_2"
    )

    input_static = Input(
        shape=(12,),
        name="Variables_estaticas"
    )

    input_crop = Input(
        shape=(1,),
        name="Cultivo"
    )


    # ConvLSTM1D necesita:
    # (batch, time, spatial, channels)
    #
    # Convertimos:
    # ERA5       (5,4) -> (5,4,1)
    # Sentinel   (5,3) -> (5,3,1)

    x_era5 = Reshape(
        (5, 4, 1),
        name="Reshape_ERA5_ConvLSTM"
    )(input_era5)

    x_sentinel = Reshape(
        (5, 3, 1),
        name="Reshape_Sentinel_ConvLSTM"
    )(input_sentinel)


    x_era5 = ConvLSTM1D(
        filters=8,
        kernel_size=2,
        padding="same",
        activation="tanh",
        recurrent_activation="sigmoid",
        kernel_regularizer=l2(L2_REG),
        recurrent_regularizer=l2(L2_REG),
        return_sequences=False,
        name="ConvLSTM1D_ERA5"
    )(x_era5)

    x_era5 = GlobalAveragePooling1D(
        name="GAP_ConvLSTM_ERA5"
    )(x_era5)

    x_era5 = Dropout(
        DROPOUT_RATE,
        name="Dropout_ERA5"
    )(x_era5)


    x_sentinel = ConvLSTM1D(
        filters=6,
        kernel_size=2,
        padding="same",
        activation="tanh",
        recurrent_activation="sigmoid",
        kernel_regularizer=l2(L2_REG),
        recurrent_regularizer=l2(L2_REG),
        return_sequences=False,
        name="ConvLSTM1D_Sentinel"
    )(x_sentinel)

    x_sentinel = GlobalAveragePooling1D(
        name="GAP_ConvLSTM_Sentinel"
    )(x_sentinel)

    x_sentinel = Dropout(
        DROPOUT_RATE,
        name="Dropout_Sentinel"
    )(x_sentinel)


    output = construir_salida_comun(
        x_era5,
        x_sentinel,
        input_static,
        input_crop
    )


    model = Model(
        inputs=[
            input_era5,
            input_sentinel,
            input_static,
            input_crop
        ],
        outputs=output,
        name="ConvLSTM1D_multirrama"
    )

    return compilar_modelo(model)


# ============================================================
# FUNCIÓN DE COMPILACIÓN COMÚN
# ============================================================

def compilar_modelo(model):

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=LEARNING_RATE
        ),

        loss="mse",

        metrics=[
            tf.keras.metrics.MeanAbsoluteError(
                name="mae"
            ),

            tf.keras.metrics.RootMeanSquaredError(
                name="rmse"
            )
        ]
    )

    return model


# ============================================================
# 5. CREAR LAS 4 ARQUITECTURAS
# ============================================================

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED_DL)

modelos_prueba = {
    "LSTM": crear_modelo_lstm(),
    "GRU": crear_modelo_gru(),
    "CNN1D": crear_modelo_cnn1d(),
    "ConvLSTM1D": crear_modelo_convlstm1d()
}


# ============================================================
# 6. RESUMEN DE PARÁMETROS
# ============================================================

print("=" * 80)
print("PASO 4 - ARQUITECTURAS DEEP LEARNING")
print("=" * 80)

for nombre, modelo in modelos_prueba.items():

    print(
        f"\n{nombre}"
    )

    print(
        "Parámetros:",
        modelo.count_params()
    )

    print(
        "Salida:",
        modelo.output_shape
    )


print("\nPASO 4 FINALIZADO")

In [ ]:
# ============================================================
# PASO 5 - VALIDACIÓN TEMPORAL COMÚN
#          LSTM, GRU, CNN1D y ConvLSTM1D
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from tensorflow.keras.callbacks import EarlyStopping


# ============================================================
# 1. CONFIGURACIÓN COMÚN
# ============================================================

SEED_DL = 42

ANOS_VALIDACION = [
    2020,
    2021,
    2022
]

EPOCHS_MAX = 300
BATCH_SIZE = 32
PATIENCE = 25
MIN_DELTA = 1.0


# ============================================================
# 2. FUNCIONES DE CREACIÓN DE MODELOS
# ============================================================

fabricas_modelos = {
    "LSTM":
        crear_modelo_lstm,

    "GRU":
        crear_modelo_gru,

    "CNN1D":
        crear_modelo_cnn1d,

    "ConvLSTM1D":
        crear_modelo_convlstm1d
}


# ============================================================
# 3. RESULTADOS
# ============================================================

resultados_validacion_dl = []


print("=" * 90)
print("PASO 5 - VALIDACIÓN TEMPORAL DE LAS 4 ARQUITECTURAS")
print("=" * 90)


# ============================================================
# 4. BUCLE MODELO × FOLD
# ============================================================

for nombre_modelo, fabrica in fabricas_modelos.items():

    print("\n" + "#" * 90)
    print("ARQUITECTURA:", nombre_modelo)
    print("#" * 90)

    for ano_val in ANOS_VALIDACION:

        print(
            f"\nValidación {ano_val}"
        )

        # ----------------------------------------------------
        # Preprocesamiento del fold
        # ----------------------------------------------------

        fold = preparar_fold_dl_comun(
            ano_val
        )

        X_train_fold = (
            fold["X_train"]
        )

        y_train_fold = (
            fold["y_train"]
        )

        X_val_fold = (
            fold["X_val"]
        )

        y_val_fold = (
            fold["y_val"]
        )


        # ----------------------------------------------------
        # Crear modelo NUEVO
        # ----------------------------------------------------

        tf.keras.backend.clear_session()

        tf.keras.utils.set_random_seed(
            SEED_DL
        )

        modelo = fabrica()


        # ----------------------------------------------------
        # Early stopping común
        # ----------------------------------------------------

        early_stopping = EarlyStopping(
            monitor="val_loss",
            patience=PATIENCE,
            restore_best_weights=True,
            min_delta=MIN_DELTA
        )


        # ----------------------------------------------------
        # Entrenamiento
        # ----------------------------------------------------

        history = modelo.fit(
            X_train_fold,
            y_train_fold,

            validation_data=(
                X_val_fold,
                y_val_fold
            ),

            epochs=EPOCHS_MAX,
            batch_size=BATCH_SIZE,

            callbacks=[
                early_stopping
            ],

            verbose=0
        )


        # ----------------------------------------------------
        # Predicción
        # ----------------------------------------------------

        y_pred = (
            modelo.predict(
                X_val_fold,
                verbose=0
            )
            .reshape(-1)
        )


        # ----------------------------------------------------
        # Métricas
        # ----------------------------------------------------

        mae = mean_absolute_error(
            y_val_fold,
            y_pred
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_val_fold,
                y_pred
            )
        )

        r2 = r2_score(
            y_val_fold,
            y_pred
        )


        mejor_epoch = (
            np.argmin(
                history.history[
                    "val_loss"
                ]
            )
            + 1
        )


        # ----------------------------------------------------
        # Guardar resultado
        # ----------------------------------------------------

        resultados_validacion_dl.append(
            {
                "modelo":
                    nombre_modelo,

                "ano_validacion":
                    ano_val,

                "n_train":
                    fold["n_train"],

                "n_val":
                    fold["n_val"],

                "mejor_epoch":
                    mejor_epoch,

                "MAE":
                    mae,

                "RMSE":
                    rmse,

                "R2":
                    r2
            }
        )


        print(
            f"MAE  = {mae:.2f}"
        )

        print(
            f"RMSE = {rmse:.2f}"
        )

        print(
            f"R²   = {r2:.3f}"
        )

        print(
            f"Mejor epoch = {mejor_epoch}"
        )


# ============================================================
# 5. TABLA DETALLADA
# ============================================================

resultados_validacion_dl_df = (
    pd.DataFrame(
        resultados_validacion_dl
    )
)

print("\n" + "=" * 90)
print("RESULTADOS DETALLADOS")
print("=" * 90)

display(
    resultados_validacion_dl_df
    .round(3)
)


# ============================================================
# 6. RESUMEN POR ARQUITECTURA
# ============================================================

resumen_validacion_dl = (
    resultados_validacion_dl_df
    .groupby("modelo")
    .agg(
        MAE_medio=(
            "MAE",
            "mean"
        ),

        MAE_std=(
            "MAE",
            "std"
        ),

        RMSE_medio=(
            "RMSE",
            "mean"
        ),

        RMSE_std=(
            "RMSE",
            "std"
        ),

        R2_medio=(
            "R2",
            "mean"
        ),

        R2_std=(
            "R2",
            "std"
        ),

        R2_min=(
            "R2",
            "min"
        ),

        R2_max=(
            "R2",
            "max"
        ),

        epoch_medio=(
            "mejor_epoch",
            "mean"
        )
    )
    .reset_index()
    .sort_values(
        "RMSE_medio"
    )
    .reset_index(
        drop=True
    )
)


print("\n" + "=" * 90)
print("RESUMEN DE VALIDACIÓN TEMPORAL")
print("=" * 90)

display(
    resumen_validacion_dl
    .round(3)
)


# ============================================================
# 7. COMPARACIÓN CON MODELOS ML YA EXISTENTES
# ============================================================

comparacion_ml_dl_validacion = pd.DataFrame(
    [
        {
            "modelo":
                "RF_XGB_50_50",

            "familia":
                "Machine Learning",

            "RMSE_medio":
                682.161,

            "R2_medio":
                0.433
        },

        {
            "modelo":
                "RF70_XGB30",

            "familia":
                "Machine Learning",

            "RMSE_medio":
                686.803,

            "R2_medio":
                0.446
        },

        {
            "modelo":
                "XGB_tuned",

            "familia":
                "Machine Learning",

            "RMSE_medio":
                700.216,

            "R2_medio":
                0.356
        },

        {
            "modelo":
                "RF_tuned",

            "familia":
                "Machine Learning",

            "RMSE_medio":
                705.102,

            "R2_medio":
                0.449
        }
    ]
)

for _, fila in (
    resumen_validacion_dl.iterrows()
):

    nueva_fila = pd.DataFrame(
        [
            {
                "modelo":
                    fila["modelo"],

                "familia":
                    "Deep Learning",

                "RMSE_medio":
                    fila["RMSE_medio"],

                "R2_medio":
                    fila["R2_medio"]
            }
        ]
    )

    comparacion_ml_dl_validacion = (
        pd.concat(
            [
                comparacion_ml_dl_validacion,
                nueva_fila
            ],
            ignore_index=True
        )
    )


comparacion_ml_dl_validacion = (
    comparacion_ml_dl_validacion
    .sort_values(
        "RMSE_medio"
    )
    .reset_index(drop=True)
)


print("\n" + "=" * 90)
print("COMPARACIÓN ML vs DEEP LEARNING - VALIDACIÓN")
print("=" * 90)

display(
    comparacion_ml_dl_validacion
    .round(3)
)


# ============================================================
# 8. GUARDAR RESULTADOS
# ============================================================

F_VALIDACION_DETALLE = (
    f"{DL_DIR}/"
    "paso05_validacion_temporal_DL_detalle.csv"
)

F_VALIDACION_RESUMEN = (
    f"{DL_DIR}/"
    "paso05_validacion_temporal_DL_resumen.csv"
)

F_COMPARACION_ML_DL = (
    f"{DL_DIR}/"
    "paso05_comparacion_ML_DL_validacion.csv"
)

resultados_validacion_dl_df.to_csv(
    F_VALIDACION_DETALLE,
    index=False
)

resumen_validacion_dl.to_csv(
    F_VALIDACION_RESUMEN,
    index=False
)

comparacion_ml_dl_validacion.to_csv(
    F_COMPARACION_ML_DL,
    index=False
)


# ============================================================
# 9. CONTROL METODOLÓGICO
# ============================================================

print("\n" + "=" * 90)
print("CONTROL METODOLÓGICO")
print("=" * 90)

print(
    "Folds:",
    ANOS_VALIDACION
)

print(
    "Mismo preprocesamiento:",
    True
)

print(
    "Mismo batch size:",
    BATCH_SIZE
)

print(
    "Mismo EarlyStopping:",
    True
)

print(
    "2023 utilizado:",
    False
)

print(
    "\nLa arquitectura DL final se seleccionará "
    "EXCLUSIVAMENTE a partir de esta validación."
)

print("\nPASO 5 FINALIZADO")

In [ ]:
# ============================================================
# PASO 6 - ENTRENAMIENTO FINAL LSTM
#          TRAIN 2017-2022
#          TEST 2023 TODAVÍA BLOQUEADO
# ============================================================

import os
import joblib
import numpy as np
import tensorflow as tf

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler


# ============================================================
# 1. CONFIGURACIÓN FINAL
# ============================================================

SEED_DL = 42
EPOCHS_FINAL_LSTM = 98
BATCH_SIZE = 32

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED_DL)


# ============================================================
# 2. SEPARACIÓN DEFINITIVA
# ============================================================

mask_dev = years <= 2022
mask_test = years == 2023

print("=" * 90)
print("PASO 6 - ENTRENAMIENTO FINAL LSTM")
print("=" * 90)

print("\nDESARROLLO:")
print("Años:", sorted(np.unique(years[mask_dev])))
print("n =", mask_dev.sum())

print("\nTEST INDEPENDIENTE:")
print("Año: 2023")
print("n =", mask_test.sum())

print("\nÉpocas finales:", EPOCHS_FINAL_LSTM)


# ============================================================
# 3. ERA5-LAND
# ============================================================

era5_dev = X_era5_seq[mask_dev].astype(np.float32)
era5_test = X_era5_seq[mask_test].astype(np.float32)

era5_dev_2d = era5_dev.reshape(-1, 4)
era5_test_2d = era5_test.reshape(-1, 4)

scaler_era5_final = StandardScaler()

era5_dev_scaled = scaler_era5_final.fit_transform(
    era5_dev_2d
)

era5_test_scaled = scaler_era5_final.transform(
    era5_test_2d
)

X_era5_dev_final = (
    era5_dev_scaled
    .reshape(era5_dev.shape)
    .astype(np.float32)
)

X_era5_test_final = (
    era5_test_scaled
    .reshape(era5_test.shape)
    .astype(np.float32)
)


# ============================================================
# 4. SENTINEL-2
# ============================================================

sentinel_dev = X_sentinel_seq[mask_dev].astype(np.float32)
sentinel_test = X_sentinel_seq[mask_test].astype(np.float32)

sentinel_dev_2d = sentinel_dev.reshape(-1, 3)
sentinel_test_2d = sentinel_test.reshape(-1, 3)


# ------------------------------------------------------------
# Imputación SOLO con 2017-2022
# ------------------------------------------------------------

imputer_sentinel_final = SimpleImputer(
    strategy="median"
)

sentinel_dev_imp = imputer_sentinel_final.fit_transform(
    sentinel_dev_2d
)

sentinel_test_imp = imputer_sentinel_final.transform(
    sentinel_test_2d
)


# ------------------------------------------------------------
# Escalado SOLO con 2017-2022
# ------------------------------------------------------------

scaler_sentinel_final = StandardScaler()

sentinel_dev_scaled = scaler_sentinel_final.fit_transform(
    sentinel_dev_imp
)

sentinel_test_scaled = scaler_sentinel_final.transform(
    sentinel_test_imp
)

X_sentinel_dev_final = (
    sentinel_dev_scaled
    .reshape(sentinel_dev.shape)
    .astype(np.float32)
)

X_sentinel_test_final = (
    sentinel_test_scaled
    .reshape(sentinel_test.shape)
    .astype(np.float32)
)


# ============================================================
# 5. VARIABLES ESTÁTICAS / AGREGADAS
# ============================================================

static_dev = X_static[mask_dev].astype(np.float32)
static_test = X_static[mask_test].astype(np.float32)


# ------------------------------------------------------------
# Imputación
# ------------------------------------------------------------

imputer_static_final = SimpleImputer(
    strategy="median"
)

static_dev_imp = imputer_static_final.fit_transform(
    static_dev
)

static_test_imp = imputer_static_final.transform(
    static_test
)


# ------------------------------------------------------------
# Escalado
# ------------------------------------------------------------

scaler_static_final = StandardScaler()

X_static_dev_final = (
    scaler_static_final
    .fit_transform(static_dev_imp)
    .astype(np.float32)
)

X_static_test_final = (
    scaler_static_final
    .transform(static_test_imp)
    .astype(np.float32)
)


# ============================================================
# 6. CULTIVO
# ============================================================

X_crop_dev_final = (
    X_crop[mask_dev]
    .astype(np.float32)
)

X_crop_test_final = (
    X_crop[mask_test]
    .astype(np.float32)
)


# ============================================================
# 7. TARGET
# ============================================================

y_dev_final = (
    y[mask_dev]
    .astype(np.float32)
)

y_test_final = (
    y[mask_test]
    .astype(np.float32)
)


# ============================================================
# 8. CREAR LSTM NUEVA
# ============================================================

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED_DL)

modelo_lstm_final = crear_modelo_lstm()


# ============================================================
# 9. ENTRENAMIENTO FINAL
# ============================================================

history_lstm_final = modelo_lstm_final.fit(
    [
        X_era5_dev_final,
        X_sentinel_dev_final,
        X_static_dev_final,
        X_crop_dev_final
    ],

    y_dev_final,

    epochs=EPOCHS_FINAL_LSTM,
    batch_size=BATCH_SIZE,

    shuffle=True,

    verbose=0
)


# ============================================================
# 10. GUARDAR MODELO Y PREPROCESADORES
# ============================================================

MODELOS_DL_DIR = (
    f"{DL_DIR}/Modelos"
)

os.makedirs(
    MODELOS_DL_DIR,
    exist_ok=True
)

F_MODELO_LSTM = (
    f"{MODELOS_DL_DIR}/"
    "LSTM_multirrama_final.keras"
)

modelo_lstm_final.save(
    F_MODELO_LSTM
)


joblib.dump(
    scaler_era5_final,
    f"{MODELOS_DL_DIR}/scaler_ERA5_LSTM.joblib"
)

joblib.dump(
    imputer_sentinel_final,
    f"{MODELOS_DL_DIR}/imputer_Sentinel_LSTM.joblib"
)

joblib.dump(
    scaler_sentinel_final,
    f"{MODELOS_DL_DIR}/scaler_Sentinel_LSTM.joblib"
)

joblib.dump(
    imputer_static_final,
    f"{MODELOS_DL_DIR}/imputer_Static_LSTM.joblib"
)

joblib.dump(
    scaler_static_final,
    f"{MODELOS_DL_DIR}/scaler_Static_LSTM.joblib"
)


# ============================================================
# 11. RESULTADO FINAL
# ============================================================

print("\n" + "=" * 90)
print("LSTM FINAL ENTRENADA")
print("=" * 90)

print("\nTrain:")
print("2017-2022")

print("Observaciones:")
print(mask_dev.sum())

print("\nÉpocas:")
print(EPOCHS_FINAL_LSTM)

print("\nParámetros:")
print(modelo_lstm_final.count_params())

print("\nModelo guardado:")
print(F_MODELO_LSTM)

print("\n2023 utilizado para:")
print("Entrenamiento       : NO")
print("Selección modelo    : NO")
print("Selección epochs    : NO")
print("Evaluación          : TODAVÍA NO")

print("\nPASO 6 FINALIZADO")

In [ ]:
# ============================================================
# PASO 7 - TEST INDEPENDIENTE 2023
#          LSTM MULTIRRAMA FINAL
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ============================================================
# 1. PREDICCIÓN ÚNICA SOBRE 2023
# ============================================================

y_pred_2023_lstm = (
    modelo_lstm_final.predict(
        [
            X_era5_test_final,
            X_sentinel_test_final,
            X_static_test_final,
            X_crop_test_final
        ],
        verbose=0
    )
    .reshape(-1)
)


# ============================================================
# 2. MÉTRICAS GLOBALES
# ============================================================

mae_2023 = mean_absolute_error(
    y_test_final,
    y_pred_2023_lstm
)

rmse_2023 = np.sqrt(
    mean_squared_error(
        y_test_final,
        y_pred_2023_lstm
    )
)

r2_2023 = r2_score(
    y_test_final,
    y_pred_2023_lstm
)

sesgo_2023 = np.mean(
    y_pred_2023_lstm - y_test_final
)


resultado_global_2023 = pd.DataFrame(
    [{
        "modelo": "LSTM_multirrama",
        "n_test": len(y_test_final),
        "MAE": mae_2023,
        "RMSE": rmse_2023,
        "R2": r2_2023,
        "sesgo_pred_menos_obs": sesgo_2023
    }]
)


print("=" * 90)
print("PASO 7 - TEST INDEPENDIENTE 2023")
print("=" * 90)

print("\nRESULTADO GLOBAL")
display(
    resultado_global_2023.round(3)
)


# ============================================================
# 3. DATAFRAME DE PREDICCIONES
# ============================================================

df_test_2023 = (
    df.loc[mask_test, [
        "provincia_estandar",
        "ano",
        "cultivo",
        "rendimiento_kg_ha"
    ]]
    .copy()
    .reset_index(drop=True)
)

df_test_2023["predicho_kg_ha"] = (
    y_pred_2023_lstm
)

df_test_2023["residuo_kg_ha"] = (
    df_test_2023["rendimiento_kg_ha"]
    - df_test_2023["predicho_kg_ha"]
)

df_test_2023["error_absoluto_kg_ha"] = (
    np.abs(
        df_test_2023["residuo_kg_ha"]
    )
)


# ============================================================
# 4. RESULTADOS POR CULTIVO
# ============================================================

resultados_cultivo = []

for cultivo in df_test_2023["cultivo"].unique():

    sub = df_test_2023[
        df_test_2023["cultivo"] == cultivo
    ].copy()

    y_obs = sub["rendimiento_kg_ha"].values
    y_pred = sub["predicho_kg_ha"].values

    mae = mean_absolute_error(
        y_obs,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_obs,
            y_pred
        )
    )

    r2 = r2_score(
        y_obs,
        y_pred
    )

    sesgo = np.mean(
        y_pred - y_obs
    )

    resultados_cultivo.append(
        {
            "modelo":
                "LSTM_multirrama",

            "cultivo":
                cultivo,

            "n":
                len(sub),

            "observado_medio":
                np.mean(y_obs),

            "predicho_medio":
                np.mean(y_pred),

            "sesgo":
                sesgo,

            "MAE":
                mae,

            "RMSE":
                rmse,

            "R2":
                r2
        }
    )


resultados_cultivo_df = pd.DataFrame(
    resultados_cultivo
)


print("\n" + "=" * 90)
print("RESULTADOS 2023 POR CULTIVO")
print("=" * 90)

display(
    resultados_cultivo_df.round(3)
)


# ============================================================
# 5. COMPARACIÓN VALIDACIÓN vs TEST
# ============================================================

lstm_validacion = (
    resumen_validacion_dl[
        resumen_validacion_dl["modelo"] == "LSTM"
    ]
    .iloc[0]
)

comparacion_val_test = pd.DataFrame(
    [{
        "modelo":
            "LSTM_multirrama",

        "RMSE_validacion_2020_2022":
            lstm_validacion["RMSE_medio"],

        "R2_validacion_2020_2022":
            lstm_validacion["R2_medio"],

        "RMSE_test_2023":
            rmse_2023,

        "R2_test_2023":
            r2_2023,

        "delta_RMSE":
            rmse_2023
            - lstm_validacion["RMSE_medio"],

        "delta_R2":
            r2_2023
            - lstm_validacion["R2_medio"]
    }]
)


print("\n" + "=" * 90)
print("VALIDACIÓN TEMPORAL vs TEST INDEPENDIENTE")
print("=" * 90)

display(
    comparacion_val_test.round(3)
)


# ============================================================
# 6. GUARDAR RESULTADOS
# ============================================================

F_TEST_GLOBAL = (
    f"{DL_DIR}/"
    "paso07_LSTM_test2023_global.csv"
)

F_TEST_CULTIVO = (
    f"{DL_DIR}/"
    "paso07_LSTM_test2023_por_cultivo.csv"
)

F_PREDICCIONES = (
    f"{DL_DIR}/"
    "paso07_LSTM_predicciones_2023.csv"
)

F_VAL_TEST = (
    f"{DL_DIR}/"
    "paso07_LSTM_validacion_vs_test.csv"
)


resultado_global_2023.to_csv(
    F_TEST_GLOBAL,
    index=False
)

resultados_cultivo_df.to_csv(
    F_TEST_CULTIVO,
    index=False
)

df_test_2023.to_csv(
    F_PREDICCIONES,
    index=False
)

comparacion_val_test.to_csv(
    F_VAL_TEST,
    index=False
)


# ============================================================
# 7. CONTROL METODOLÓGICO FINAL
# ============================================================

print("\n" + "=" * 90)
print("CONTROL METODOLÓGICO")
print("=" * 90)

print("Arquitectura seleccionada: LSTM multirrama")
print("Selección realizada con: validación 2020-2022")
print("Entrenamiento final: 2017-2022")
print("n train:", len(y_dev_final))

print("\nTest independiente: 2023")
print("n test:", len(y_test_final))

print("\nA partir de este punto:")
print("- NO modificar arquitectura usando 2023.")
print("- NO ajustar hiperparámetros usando 2023.")
print("- NO seleccionar variables usando 2023.")
print("- NO reentrenar buscando mejorar las métricas de 2023.")

print(
    "\n2023 se ha utilizado exclusivamente "
    "como evaluación final independiente."
)

print("\nPASO 7 FINALIZADO")

In [ ]:
# ============================================================
# PASO 8 - COMPARACIÓN FINAL ML vs DEEP LEARNING
# ============================================================

import pandas as pd
import numpy as np
import os

# ============================================================
# 1. RESULTADOS FINALES TEST 2023
# ============================================================

comparacion_final = pd.DataFrame([
    {
        "modelo": "LSTM_multirrama",
        "familia": "Deep Learning",
        "MAE": 572.012,
        "RMSE": 714.903,
        "R2": 0.527
    },
    {
        "modelo": "XGB_tuned",
        "familia": "Machine Learning",
        "MAE": 926.008,
        "RMSE": 1102.817,
        "R2": -0.126
    },
    {
        "modelo": "RF_XGB_50_50",
        "familia": "Ensemble ML",
        "MAE": 949.490,
        "RMSE": 1118.798,
        "R2": -0.159
    },
    {
        "modelo": "RF70_XGB30",
        "familia": "Ensemble ML",
        "MAE": 961.305,
        "RMSE": 1129.432,
        "R2": -0.181
    },
    {
        "modelo": "RF_tuned",
        "familia": "Machine Learning",
        "MAE": 980.285,
        "RMSE": 1149.732,
        "R2": -0.224
    }
])

comparacion_final = (
    comparacion_final
    .sort_values("RMSE")
    .reset_index(drop=True)
)

comparacion_final.insert(
    0,
    "ranking",
    np.arange(1, len(comparacion_final) + 1)
)

print("=" * 90)
print("PASO 8 - COMPARACIÓN FINAL ML vs DEEP LEARNING")
print("=" * 90)

display(comparacion_final.round(3))


# ============================================================
# 2. MEJORA DE LSTM RESPECTO AL MEJOR MODELO ML
# ============================================================

lstm = comparacion_final[
    comparacion_final["modelo"] == "LSTM_multirrama"
].iloc[0]

mejor_ml = (
    comparacion_final[
        comparacion_final["familia"] != "Deep Learning"
    ]
    .sort_values("RMSE")
    .iloc[0]
)

reduccion_rmse = (
    (mejor_ml["RMSE"] - lstm["RMSE"])
    / mejor_ml["RMSE"]
    * 100
)

reduccion_mae = (
    (mejor_ml["MAE"] - lstm["MAE"])
    / mejor_ml["MAE"]
    * 100
)

print("\n" + "=" * 90)
print("LSTM vs MEJOR MODELO ML EN TEST 2023")
print("=" * 90)

print("Mejor ML:", mejor_ml["modelo"])

print(
    f"Reducción de RMSE con LSTM: "
    f"{reduccion_rmse:.2f}%"
)

print(
    f"Reducción de MAE con LSTM: "
    f"{reduccion_mae:.2f}%"
)

print(
    f"Diferencia de R²: "
    f"{lstm['R2'] - mejor_ml['R2']:.3f}"
)


# ============================================================
# 3. GUARDAR TABLA FINAL
# ============================================================

F_COMPARACION_FINAL = (
    f"{DL_DIR}/"
    "paso08_comparacion_final_ML_DL_test2023.csv"
)

comparacion_final.to_csv(
    F_COMPARACION_FINAL,
    index=False
)

print("\nArchivo guardado:")
print(F_COMPARACION_FINAL)


# ============================================================
# 4. CONTROL METODOLÓGICO
# ============================================================

print("\n" + "=" * 90)
print("CONCLUSIÓN EXPERIMENTAL")
print("=" * 90)

print(
    "Mejor modelo final:",
    comparacion_final.iloc[0]["modelo"]
)

print(
    "RMSE test 2023:",
    round(comparacion_final.iloc[0]["RMSE"], 3)
)

print(
    "R² test 2023:",
    round(comparacion_final.iloc[0]["R2"], 3)
)

print("\nNo se realizará nuevo tuning utilizando 2023.")
print("No se modificarán modelos a partir del resultado del test.")
print("2023 permanece como evaluación final independiente.")

print("\nPASO 8 FINALIZADO")

# Graficos finales modelo LSTM

In [ ]:
# ============================================================
# PASO 9 - FIGURAS FINALES DEL MODELO LSTM
# TEST INDEPENDIENTE 2023
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. DIRECTORIO DE FIGURAS
# ------------------------------------------------------------

FIG_DIR = (
    "/content/TFM_Rendimiento_cultivos/"
    "05_Resultados/Deep_Learning_Comparacion/Figuras"
)

os.makedirs(FIG_DIR, exist_ok=True)

print("=" * 90)
print("PASO 9 - DIAGNÓSTICO GRÁFICO LSTM - TEST 2023")
print("=" * 90)


# ------------------------------------------------------------
# 2. LOCALIZAR DATAFRAME DE PREDICCIONES DEL PASO 7
# ------------------------------------------------------------

# Buscamos entre nombres razonables que puedan haberse utilizado
candidatos = [
    "df_test_2023",
    "df_pred_2023",
    "predicciones_2023",
    "resultados_2023",
    "df_resultados_2023"
]

df_plot = None
nombre_encontrado = None

for nombre in candidatos:
    if nombre in globals():
        obj = globals()[nombre]
        if isinstance(obj, pd.DataFrame):
            df_plot = obj.copy()
            nombre_encontrado = nombre
            break

if df_plot is None:
    raise NameError(
        "No encuentro el DataFrame de predicciones del Paso 7. "
        "No continúes todavía: muéstrame las variables disponibles "
        "o vuelve a ejecutar el Paso 7."
    )

print("\nDataFrame encontrado:", nombre_encontrado)
print("Dimensiones:", df_plot.shape)
print("Columnas:")
print(df_plot.columns.tolist())


# ------------------------------------------------------------
# 3. IDENTIFICAR COLUMNAS
# ------------------------------------------------------------

obs_candidates = [
    "rendimiento_kg_ha",
    "observado_kg_ha",
    "y_true",
    "observado"
]

pred_candidates = [
    "predicho_kg_ha",
    "prediccion_kg_ha",
    "y_pred",
    "predicho"
]

obs_col = next(
    (c for c in obs_candidates if c in df_plot.columns),
    None
)

pred_col = next(
    (c for c in pred_candidates if c in df_plot.columns),
    None
)

if obs_col is None or pred_col is None:
    raise ValueError(
        "No se han podido identificar las columnas observado/predicho. "
        f"Columnas disponibles: {df_plot.columns.tolist()}"
    )

print("\nVariable observada:", obs_col)
print("Variable predicha :", pred_col)


# ------------------------------------------------------------
# 4. CALCULAR RESIDUOS
# residuo = observado - predicho
# ------------------------------------------------------------

df_plot["residuo_kg_ha"] = (
    df_plot[obs_col] - df_plot[pred_col]
)

df_plot["error_absoluto_kg_ha"] = (
    df_plot["residuo_kg_ha"].abs()
)


# ============================================================
# FIGURA 9A
# OBSERVADO vs PREDICHO
# ============================================================

fig, ax = plt.subplots(figsize=(7, 7))

# Si existe cultivo, representar cada cultivo por separado
if "cultivo" in df_plot.columns:

    for cultivo, grupo in df_plot.groupby("cultivo"):

        etiqueta = {
            "barley": "Cebada",
            "common_soft_wheat": "Trigo blando"
        }.get(cultivo, cultivo)

        ax.scatter(
            grupo[obs_col],
            grupo[pred_col],
            alpha=0.75,
            s=45,
            label=etiqueta
        )

else:

    ax.scatter(
        df_plot[obs_col],
        df_plot[pred_col],
        alpha=0.75,
        s=45
    )


# Línea ideal y = x
minimo = min(
    df_plot[obs_col].min(),
    df_plot[pred_col].min()
)

maximo = max(
    df_plot[obs_col].max(),
    df_plot[pred_col].max()
)

ax.plot(
    [minimo, maximo],
    [minimo, maximo],
    linestyle="--",
    linewidth=1.5,
    label="Predicción perfecta"
)

ax.set_xlabel("Rendimiento observado (kg/ha)")
ax.set_ylabel("Rendimiento predicho (kg/ha)")
ax.set_title(
    "LSTM multirrama — observado vs. predicho\n"
    "Test independiente 2023"
)

ax.grid(alpha=0.25)
ax.legend()

fig.tight_layout()

F_FIG_9A = os.path.join(
    FIG_DIR,
    "Figura_LSTM_observado_vs_predicho_2023.png"
)

fig.savefig(
    F_FIG_9A,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# FIGURA 9B
# RESIDUOS vs PREDICCIÓN
# ============================================================

fig, ax = plt.subplots(figsize=(8, 6))

if "cultivo" in df_plot.columns:

    for cultivo, grupo in df_plot.groupby("cultivo"):

        etiqueta = {
            "barley": "Cebada",
            "common_soft_wheat": "Trigo blando"
        }.get(cultivo, cultivo)

        ax.scatter(
            grupo[pred_col],
            grupo["residuo_kg_ha"],
            alpha=0.75,
            s=45,
            label=etiqueta
        )

else:

    ax.scatter(
        df_plot[pred_col],
        df_plot["residuo_kg_ha"],
        alpha=0.75,
        s=45
    )


ax.axhline(
    0,
    linestyle="--",
    linewidth=1.5
)

ax.set_xlabel("Rendimiento predicho (kg/ha)")
ax.set_ylabel("Residuo: observado − predicho (kg/ha)")
ax.set_title(
    "LSTM multirrama — análisis de residuos\n"
    "Test independiente 2023"
)

ax.grid(alpha=0.25)

if "cultivo" in df_plot.columns:
    ax.legend()

fig.tight_layout()

F_FIG_9B = os.path.join(
    FIG_DIR,
    "Figura_LSTM_residuos_2023.png"
)

fig.savefig(
    F_FIG_9B,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 5. RESUMEN NUMÉRICO DE RESIDUOS
# ============================================================

print("\n" + "=" * 90)
print("RESUMEN DE RESIDUOS")
print("=" * 90)

print(
    df_plot["residuo_kg_ha"]
    .describe()
    .round(3)
)


# ------------------------------------------------------------
# 6. MAYORES ERRORES
# ------------------------------------------------------------

columnas_mostrar = [
    c for c in [
        "provincia_estandar",
        "ano",
        "cultivo",
        obs_col,
        pred_col,
        "residuo_kg_ha",
        "error_absoluto_kg_ha"
    ]
    if c in df_plot.columns
]

mayores_errores = (
    df_plot[columnas_mostrar]
    .sort_values(
        "error_absoluto_kg_ha",
        ascending=False
    )
    .head(10)
)

print("\n10 OBSERVACIONES CON MAYOR ERROR ABSOLUTO")
display(mayores_errores.round(2))


# ------------------------------------------------------------
# 7. GUARDAR RESULTADOS
# ------------------------------------------------------------

F_PRED = os.path.join(
    FIG_DIR,
    "predicciones_LSTM_test2023.csv"
)

df_plot.to_csv(
    F_PRED,
    index=False
)

print("\n" + "=" * 90)
print("ARCHIVOS GENERADOS")
print("=" * 90)

print(F_FIG_9A)
print(F_FIG_9B)
print(F_PRED)

print("\nModelo reentrenado: NO")
print("Test utilizado para modificar el modelo: NO")
print("Análisis: diagnóstico post hoc del modelo final.")

print("\nPASO 9 FINALIZADO")

In [ ]:
# ============================================================
# PASO 10 - FIGURAS COMPARATIVAS FINALES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

FIG_DIR = (
    "/content/TFM_Rendimiento_cultivos/"
    "05_Resultados/Deep_Learning_Comparacion/Figuras"
)

os.makedirs(FIG_DIR, exist_ok=True)

print("=" * 90)
print("PASO 10 - FIGURAS COMPARATIVAS FINALES")
print("=" * 90)


# ============================================================
# 1. COMPARACIÓN DE ARQUITECTURAS DL
# VALIDACIÓN TEMPORAL 2020-2022
# ============================================================

dl_validacion = pd.DataFrame({
    "modelo": [
        "LSTM",
        "GRU",
        "CNN1D",
        "ConvLSTM1D"
    ],
    "RMSE_medio": [
        769.075,
        812.265,
        828.887,
        984.376
    ],
    "RMSE_std": [
        213.396,
        238.209,
        201.153,
        237.430
    ],
    "R2_medio": [
        0.438,
        0.366,
        0.324,
        0.036
    ],
    "R2_std": [
        0.126,
        0.217,
        0.199,
        0.317
    ]
})


# ------------------------------------------------------------
# FIGURA 10A - RMSE DL
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(8, 6))

ax.bar(
    dl_validacion["modelo"],
    dl_validacion["RMSE_medio"],
    yerr=dl_validacion["RMSE_std"],
    capsize=5
)

ax.set_ylabel("RMSE medio (kg/ha)")
ax.set_xlabel("Arquitectura")
ax.set_title(
    "Comparación de arquitecturas Deep Learning\n"
    "Validación temporal 2020–2022"
)

ax.grid(
    axis="y",
    alpha=0.25
)

fig.tight_layout()

F_10A = os.path.join(
    FIG_DIR,
    "Figura_comparacion_DL_RMSE_validacion.png"
)

fig.savefig(
    F_10A,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# FIGURA 10B - R2 DL
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(8, 6))

ax.bar(
    dl_validacion["modelo"],
    dl_validacion["R2_medio"],
    yerr=dl_validacion["R2_std"],
    capsize=5
)

ax.axhline(
    0,
    linewidth=1
)

ax.set_ylabel("R² medio")
ax.set_xlabel("Arquitectura")
ax.set_title(
    "Capacidad explicativa de las arquitecturas Deep Learning\n"
    "Validación temporal 2020–2022"
)

ax.grid(
    axis="y",
    alpha=0.25
)

fig.tight_layout()

F_10B = os.path.join(
    FIG_DIR,
    "Figura_comparacion_DL_R2_validacion.png"
)

fig.savefig(
    F_10B,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 2. COMPARACIÓN FINAL TEST INDEPENDIENTE 2023
# ============================================================

modelos_test = pd.DataFrame({
    "modelo": [
        "LSTM",
        "XGBoost",
        "RF-XGB 50/50",
        "RF70-XGB30",
        "Random Forest"
    ],
    "RMSE": [
        714.903,
        1102.817,
        1118.798,
        1129.432,
        1149.732
    ],
    "R2": [
        0.527,
        -0.126,
        -0.159,
        -0.181,
        -0.224
    ]
})


# ------------------------------------------------------------
# FIGURA 10C - RMSE TEST 2023
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(9, 6))

bars = ax.bar(
    modelos_test["modelo"],
    modelos_test["RMSE"]
)

ax.set_ylabel("RMSE (kg/ha)")
ax.set_xlabel("Modelo")
ax.set_title(
    "Comparación final de modelos\n"
    "Test independiente 2023"
)

ax.tick_params(
    axis="x",
    rotation=20
)

ax.grid(
    axis="y",
    alpha=0.25
)

# Mostrar valores
for bar, valor in zip(
    bars,
    modelos_test["RMSE"]
):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 15,
        f"{valor:.0f}",
        ha="center",
        va="bottom"
    )

fig.tight_layout()

F_10C = os.path.join(
    FIG_DIR,
    "Figura_comparacion_modelos_RMSE_test2023.png"
)

fig.savefig(
    F_10C,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# FIGURA 10D - R2 TEST 2023
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(9, 6))

bars = ax.bar(
    modelos_test["modelo"],
    modelos_test["R2"]
)

ax.axhline(
    0,
    linewidth=1
)

ax.set_ylabel("R²")
ax.set_xlabel("Modelo")
ax.set_title(
    "Comparación de capacidad predictiva\n"
    "Test independiente 2023"
)

ax.tick_params(
    axis="x",
    rotation=20
)

ax.grid(
    axis="y",
    alpha=0.25
)

for bar, valor in zip(
    bars,
    modelos_test["R2"]
):
    desplazamiento = 0.02 if valor >= 0 else -0.04

    ax.text(
        bar.get_x() + bar.get_width()/2,
        valor + desplazamiento,
        f"{valor:.3f}",
        ha="center",
        va="bottom" if valor >= 0 else "top"
    )

fig.tight_layout()

F_10D = os.path.join(
    FIG_DIR,
    "Figura_comparacion_modelos_R2_test2023.png"
)

fig.savefig(
    F_10D,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 3. GUARDAR TABLAS
# ============================================================

dl_validacion.to_csv(
    os.path.join(
        FIG_DIR,
        "tabla_comparacion_arquitecturas_DL.csv"
    ),
    index=False
)

modelos_test.to_csv(
    os.path.join(
        FIG_DIR,
        "tabla_comparacion_modelos_test2023.csv"
    ),
    index=False
)


# ============================================================
# 4. CONTROL
# ============================================================

print("\n" + "=" * 90)
print("ARCHIVOS GENERADOS")
print("=" * 90)

print(F_10A)
print(F_10B)
print(F_10C)
print(F_10D)

print("\nInterpretación:")
print(
    "Selección arquitectura -> validación temporal 2020-2022."
)
print(
    "Evaluación final -> test independiente 2023."
)
print(
    "2023 NO utilizado para seleccionar la arquitectura."
)

print("\nPASO 10 FINALIZADO")

# Guardar configuración final

In [ ]:
# ============================================================
# PASO 11 - CONFIGURACIÓN DE INFERENCIA LSTM FINAL
# ============================================================

import os
import json
import joblib

# ============================================================
# 1. DIRECTORIO
# ============================================================

INFER_DIR = (
    f"{DL_DIR}/Inferencia_LSTM"
)

os.makedirs(
    INFER_DIR,
    exist_ok=True
)

# ============================================================
# 2. CONFIGURACIÓN
# ============================================================

config_inferencia_lstm = {

    "modelo": "LSTM_multirrama_final",

    "periodos_era5": [
        "ene_feb",
        "mar_abr",
        "may_jun",
        "jul_ago",
        "sep_dic"
    ],

    "variables_era5": [
        "temp_media_c",
        "temp_rocio_c",
        "precipitacion_mm",
        "radiacion_mj_m2"
    ],

    "periodos_sentinel": [
        "jan_feb",
        "mar_apr",
        "may_jun",
        "jul",
        "nov_dec"
    ],

    "indices_sentinel": [
        "NDVI",
        "EVI",
        "NDMI"
    ],

    "variables_static": [
        "clay_pct_0_30cm",
        "sand_pct_0_30cm",
        "soc_g_kg_0_30cm",
        "ph_0_30cm",
        "bdod_g_cm3_0_30cm",

        "n_recintos",
        "superficie_eurocrops_ha",
        "superficie_media_recinto_ha",

        "temp_media_c_campana",
        "temp_rocio_c_campana",
        "precipitacion_mm_campana",
        "radiacion_mj_m2_campana"
    ],

    "mapa_cultivo": {
        "barley": 0.0,
        "common_soft_wheat": 1.0
    },

    "shapes": {
        "ERA5": [5, 4],
        "Sentinel2": [5, 3],
        "Static": [12],
        "Cultivo": [1]
    },

    "target": "rendimiento_kg_ha",

    "train_period": [
        2017,
        2018,
        2019,
        2020,
        2021,
        2022
    ],

    "test_period": [
        2023
    ],

    "epochs_final": 98,

    "seed": 42
}

# ============================================================
# 3. GUARDAR CONFIGURACIÓN JSON
# ============================================================

F_CONFIG = (
    f"{INFER_DIR}/"
    "config_inferencia_LSTM.json"
)

with open(
    F_CONFIG,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        config_inferencia_lstm,
        f,
        indent=4,
        ensure_ascii=False
    )

# ============================================================
# 4. COPIAR/GUARDAR PREPROCESADORES
# ============================================================

joblib.dump(
    scaler_era5_final,
    f"{INFER_DIR}/scaler_ERA5_LSTM.joblib"
)

joblib.dump(
    imputer_sentinel_final,
    f"{INFER_DIR}/imputer_Sentinel_LSTM.joblib"
)

joblib.dump(
    scaler_sentinel_final,
    f"{INFER_DIR}/scaler_Sentinel_LSTM.joblib"
)

joblib.dump(
    imputer_static_final,
    f"{INFER_DIR}/imputer_Static_LSTM.joblib"
)

joblib.dump(
    scaler_static_final,
    f"{INFER_DIR}/scaler_Static_LSTM.joblib"
)

# ============================================================
# 5. GUARDAR REFERENCIA AL MODELO
# ============================================================

F_INFO_MODELO = (
    f"{INFER_DIR}/"
    "modelo_LSTM_path.txt"
)

with open(
    F_INFO_MODELO,
    "w",
    encoding="utf-8"
) as f:

    f.write(F_MODELO_LSTM)

# ============================================================
# 6. CONTROL FINAL
# ============================================================

print("=" * 90)
print("PASO 11 - CONFIGURACIÓN DE INFERENCIA LSTM")
print("=" * 90)

print("\nModelo:")
print(F_MODELO_LSTM)

print("\nConfiguración:")
print(F_CONFIG)

print("\nDirectorio inferencia:")
print(INFER_DIR)

print("\nArchivos:")
for archivo in sorted(os.listdir(INFER_DIR)):
    print(" -", archivo)

print("\nShapes esperados:")
print(config_inferencia_lstm["shapes"])

print("\nCultivo:")
print(config_inferencia_lstm["mapa_cultivo"])

print("\nPASO 11 FINALIZADO")

In [ ]:
# ============================================================
# PASO 12 - PRUEBA END-TO-END DE INFERENCIA
#           CARGANDO MODELO Y PREPROCESADORES DESDE DISCO
# ============================================================

import json
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf


# ============================================================
# 1. CARGAR CONFIGURACIÓN
# ============================================================

with open(
    f"{INFER_DIR}/config_inferencia_LSTM.json",
    "r",
    encoding="utf-8"
) as f:
    config_inf = json.load(f)


# ============================================================
# 2. CARGAR MODELO
# ============================================================

modelo_cargado = tf.keras.models.load_model(
    F_MODELO_LSTM
)


# ============================================================
# 3. CARGAR PREPROCESADORES
# ============================================================

scaler_era5_cargado = joblib.load(
    f"{INFER_DIR}/scaler_ERA5_LSTM.joblib"
)

imputer_sentinel_cargado = joblib.load(
    f"{INFER_DIR}/imputer_Sentinel_LSTM.joblib"
)

scaler_sentinel_cargado = joblib.load(
    f"{INFER_DIR}/scaler_Sentinel_LSTM.joblib"
)

imputer_static_cargado = joblib.load(
    f"{INFER_DIR}/imputer_Static_LSTM.joblib"
)

scaler_static_cargado = joblib.load(
    f"{INFER_DIR}/scaler_Static_LSTM.joblib"
)


# ============================================================
# 4. SELECCIONAR UNA OBSERVACIÓN REAL DE 2023
# ============================================================

idx_global = np.where(mask_test)[0][0]

fila = df.iloc[idx_global]

print("=" * 90)
print("PASO 12 - PRUEBA END-TO-END DE INFERENCIA")
print("=" * 90)

print("\nObservación seleccionada:")
print("Provincia:", fila["provincia_estandar"])
print("Cultivo:", fila["cultivo"])
print("Año:", fila["ano"])
print("Rendimiento observado:", fila["rendimiento_kg_ha"])


# ============================================================
# 5. CONSTRUIR ERA5 DE ESA OBSERVACIÓN
# ============================================================

era5_obs = np.array(
    [
        [
            fila[f"{variable}_{periodo}"]
            for variable in config_inf["variables_era5"]
        ]
        for periodo in config_inf["periodos_era5"]
    ],
    dtype=np.float32
).reshape(1, 5, 4)

era5_obs_2d = era5_obs.reshape(-1, 4)

era5_obs_scaled = (
    scaler_era5_cargado
    .transform(era5_obs_2d)
    .reshape(1, 5, 4)
    .astype(np.float32)
)


# ============================================================
# 6. CONSTRUIR SENTINEL-2
# ============================================================

sentinel_obs = np.array(
    [
        [
            fila[f"{indice}_{periodo}"]
            for indice in config_inf["indices_sentinel"]
        ]
        for periodo in config_inf["periodos_sentinel"]
    ],
    dtype=np.float32
).reshape(1, 5, 3)

sentinel_obs_2d = sentinel_obs.reshape(-1, 3)

sentinel_obs_imp = (
    imputer_sentinel_cargado
    .transform(sentinel_obs_2d)
)

sentinel_obs_scaled = (
    scaler_sentinel_cargado
    .transform(sentinel_obs_imp)
    .reshape(1, 5, 3)
    .astype(np.float32)
)


# ============================================================
# 7. CONSTRUIR VARIABLES ESTÁTICAS
# ============================================================

static_obs = np.array(
    [
        fila[col]
        for col in config_inf["variables_static"]
    ],
    dtype=np.float32
).reshape(1, -1)

static_obs_imp = (
    imputer_static_cargado
    .transform(static_obs)
)

static_obs_scaled = (
    scaler_static_cargado
    .transform(static_obs_imp)
    .astype(np.float32)
)


# ============================================================
# 8. CULTIVO
# ============================================================

crop_obs = np.array(
    [[
        config_inf["mapa_cultivo"][
            fila["cultivo"]
        ]
    ]],
    dtype=np.float32
)


# ============================================================
# 9. PREDICCIÓN DESDE ARCHIVOS CARGADOS
# ============================================================

pred_cargada = (
    modelo_cargado.predict(
        [
            era5_obs_scaled,
            sentinel_obs_scaled,
            static_obs_scaled,
            crop_obs
        ],
        verbose=0
    )
    .reshape(-1)[0]
)


# ============================================================
# 10. PREDICCIÓN ORIGINAL DEL PASO 7
# ============================================================

idx_test_local = 0

pred_original = (
    y_pred_2023_lstm[
        idx_test_local
    ]
)


# ============================================================
# 11. COMPARACIÓN
# ============================================================

diferencia = abs(
    float(pred_cargada)
    - float(pred_original)
)


print("\n" + "=" * 90)
print("COMPARACIÓN")
print("=" * 90)

print(
    "Predicción original Paso 7:",
    round(float(pred_original), 6)
)

print(
    "Predicción cargando archivos:",
    round(float(pred_cargada), 6)
)

print(
    "Diferencia absoluta:",
    diferencia
)


# ============================================================
# 12. VALIDACIÓN
# ============================================================

assert diferencia < 1e-4, (
    "La inferencia cargada no reproduce "
    "la predicción original."
)

print("\nINFERENCIA REPRODUCIBLE: SÍ")
print("Modelo y preprocesadores listos para despliegue.")

print("\nPASO 12 FINALIZADO")

# Streamlit

In [ ]:
# ============================================================
# PASO 13A - MONTAR DRIVE Y VERIFICAR DATASET
# ============================================================

import os


ROOT = "/content/TFM_Rendimiento_cultivos"
PROC = f"{ROOT}/02_Datos_procesados"

F_DATASET = (
    f"{PROC}/"
    "dataset_maestro_TFM_2017_2023_DEFINITIVO.csv"
)

print("Ruta:")
print(F_DATASET)

print("\nExiste:")
print(os.path.exists(F_DATASET))

In [ ]:
# ============================================================
# PASO 13 - DATOS PARA PRUEBA DE REPRODUCIBILIDAD EN STREAMLIT
# ============================================================

import pandas as pd

ROOT = "/content/TFM_Rendimiento_cultivos"

F_DATASET = (
    f"{ROOT}/02_Datos_procesados/"
    "dataset_maestro_TFM_2017_2023_DEFINITIVO.csv"
)

# ------------------------------------------------------------
# 1. CARGAR DATASET DEFINITIVO
# ------------------------------------------------------------

df_check = pd.read_csv(F_DATASET)

# ------------------------------------------------------------
# 2. SELECCIONAR EXACTAMENTE LA OBSERVACIÓN UTILIZADA
#    EN LA PRUEBA END-TO-END
# ------------------------------------------------------------

fila = df_check[
    (df_check["provincia_estandar"] == "A Coruña") &
    (df_check["cultivo"] == "barley") &
    (df_check["ano"] == 2023)
].copy()

print("=" * 90)
print("PASO 13 - DATOS PARA PRUEBA STREAMLIT")
print("=" * 90)

print("\nNúmero de observaciones encontradas:", len(fila))

assert len(fila) == 1, (
    f"Se esperaba exactamente 1 observación y se encontraron {len(fila)}."
)

# ------------------------------------------------------------
# 3. INFORMACIÓN DE CONTROL
# ------------------------------------------------------------

print("\nOBSERVACIÓN:")
print("Provincia:", fila["provincia_estandar"].iloc[0])
print("Cultivo:", fila["cultivo"].iloc[0])
print("Año:", fila["ano"].iloc[0])
print(
    "Rendimiento observado:",
    fila["rendimiento_kg_ha"].iloc[0],
    "kg/ha"
)

# ------------------------------------------------------------
# 4. MOSTRAR TODOS LOS PREDICTORES
# ------------------------------------------------------------

columnas_no_predictoras = [
    "provincia_estandar",
    "ano",
    "cultivo",
    "rendimiento_kg_ha"
]

predictores = [
    c for c in df_check.columns
    if c not in columnas_no_predictoras
]

print("\n" + "=" * 90)
print("PREDICTORES EXACTOS DE LA OBSERVACIÓN")
print("=" * 90)

for c in predictores:
    print(f"{c}: {fila[c].iloc[0]}")

print("\nNúmero de predictores numéricos:", len(predictores))

print("\n" + "=" * 90)
print("CONTROL")
print("=" * 90)

print("Predicción esperada del Paso 12:")
print("2103.297119 kg/ha")

print("\nEstos son los valores que deben introducirse en Streamlit")
print("para comprobar la reproducibilidad del despliegue.")

print("\nPASO 13 FINALIZADO")

# Resumen bloque Deep Learning

In [ ]:
# ============================================================
# PASO 15 - RESUMEN FINAL DEL BLOQUE DEEP LEARNING
# VERSIÓN AUTOCONTENIDA
# ============================================================

import os
import pandas as pd

ROOT = "/content/TFM_Rendimiento_cultivos"

RESUMEN_DIR = (
    f"{ROOT}/05_Resultados/"
    "Deep_Learning_Comparacion/Resumen_Final"
)

os.makedirs(RESUMEN_DIR, exist_ok=True)

print("=" * 100)
print("PASO 15 - RESUMEN FINAL DEEP LEARNING")
print("=" * 100)


# ============================================================
# 1. VALIDACIÓN TEMPORAL DE LAS CUATRO ARQUITECTURAS
#    Resultados obtenidos en el Paso 5
# ============================================================

tabla_dl_validacion = pd.DataFrame({

    "ranking_RMSE": [1, 2, 3, 4],

    "modelo": [
        "LSTM",
        "GRU",
        "CNN1D",
        "ConvLSTM1D"
    ],

    "MAE_medio": [
        642.345,
        637.808,
        661.879,
        734.030
    ],

    "MAE_std": [
        176.804,
        169.145,
        156.391,
        216.165
    ],

    "RMSE_medio": [
        769.075,
        812.265,
        828.887,
        984.376
    ],

    "RMSE_std": [
        213.396,
        238.209,
        201.153,
        237.430
    ],

    "R2_medio": [
        0.438,
        0.366,
        0.324,
        0.036
    ],

    "R2_std": [
        0.126,
        0.217,
        0.199,
        0.317
    ],

    "R2_min": [
        0.314,
        0.136,
        0.170,
        -0.165
    ],

    "R2_max": [
        0.567,
        0.567,
        0.549,
        0.402
    ],

    "epoch_medio": [
        156.333,
        166.333,
        205.333,
        197.333
    ]
})

print("\n1. ARQUITECTURAS DL - VALIDACIÓN TEMPORAL 2020-2022")
display(tabla_dl_validacion)


# ============================================================
# 2. MODELO SELECCIONADO
# ============================================================

print("\n" + "=" * 100)
print("2. ARQUITECTURA SELECCIONADA")
print("=" * 100)

print("Modelo seleccionado: LSTM multirrama")
print("RMSE medio validación: 769.075 kg/ha")
print("R² medio validación: 0.438")
print("Criterio principal: menor RMSE medio en validación temporal.")
print("2023 utilizado para seleccionar arquitectura: NO")


# ============================================================
# 3. TEST INDEPENDIENTE 2023
# ============================================================

tabla_test = pd.DataFrame([{

    "modelo": "LSTM_multirrama",
    "train": "2017-2022",
    "n_train": 572,
    "test": 2023,
    "n_test": 95,
    "epochs_final": 98,
    "parametros": 3873,
    "MAE_kg_ha": 572.012,
    "RMSE_kg_ha": 714.903,
    "R2": 0.527,
    "sesgo_pred_menos_obs_kg_ha": 195.395
}])

print("\n" + "=" * 100)
print("3. TEST INDEPENDIENTE 2023")
print("=" * 100)

display(tabla_test)


# ============================================================
# 4. RESULTADOS POR CULTIVO
# ============================================================

tabla_cultivos = pd.DataFrame({

    "cultivo": [
        "barley",
        "common_soft_wheat"
    ],

    "n": [
        47,
        48
    ],

    "observado_medio_kg_ha": [
        1346.468,
        1563.958
    ],

    "predicho_medio_kg_ha": [
        1622.202,
        1680.689
    ],

    "sesgo_kg_ha": [
        275.734,
        116.731
    ],

    "MAE_kg_ha": [
        562.492,
        581.334
    ],

    "RMSE_kg_ha": [
        713.921,
        715.863
    ],

    "R2": [
        0.409,
        0.597
    ]
})

print("\n" + "=" * 100)
print("4. RESULTADOS 2023 POR CULTIVO")
print("=" * 100)

display(tabla_cultivos)


# ============================================================
# 5. COMPARACIÓN FINAL ML vs DL
# ============================================================

comparacion_final = pd.DataFrame({

    "ranking": [1, 2, 3, 4, 5],

    "modelo": [
        "LSTM_multirrama",
        "XGB_tuned",
        "RF_XGB_50_50",
        "RF70_XGB30",
        "RF_tuned"
    ],

    "familia": [
        "Deep Learning",
        "Machine Learning",
        "Ensemble ML",
        "Ensemble ML",
        "Machine Learning"
    ],

    "MAE": [
        572.012,
        926.008,
        949.490,
        961.305,
        980.285
    ],

    "RMSE": [
        714.903,
        1102.817,
        1118.798,
        1129.432,
        1149.732
    ],

    "R2": [
        0.527,
        -0.126,
        -0.159,
        -0.181,
        -0.224
    ]
})

print("\n" + "=" * 100)
print("5. COMPARACIÓN FINAL ML vs DL - TEST 2023")
print("=" * 100)

display(comparacion_final)


# ============================================================
# 6. MEJORA RESPECTO AL MEJOR MODELO ML
# ============================================================

rmse_lstm = 714.903
mae_lstm = 572.012
r2_lstm = 0.527

rmse_xgb = 1102.817
mae_xgb = 926.008
r2_xgb = -0.126

reduccion_rmse = (
    (rmse_xgb - rmse_lstm)
    / rmse_xgb
    * 100
)

reduccion_mae = (
    (mae_xgb - mae_lstm)
    / mae_xgb
    * 100
)

diferencia_r2 = (
    r2_lstm - r2_xgb
)

print("\n" + "=" * 100)
print("6. LSTM vs MEJOR MODELO ML")
print("=" * 100)

print("Mejor modelo ML: XGB_tuned")
print(f"Reducción RMSE: {reduccion_rmse:.2f} %")
print(f"Reducción MAE : {reduccion_mae:.2f} %")
print(f"Diferencia R² : {diferencia_r2:.3f}")


# ============================================================
# 7. ARTEFACTOS DE INFERENCIA
# ============================================================

artefactos = pd.DataFrame({

    "tipo": [
        "Modelo",
        "Configuración",
        "Imputer Sentinel-2",
        "Imputer variables estáticas",
        "Scaler ERA5-Land",
        "Scaler Sentinel-2",
        "Scaler variables estáticas"
    ],

    "archivo": [
        "LSTM_multirrama_final.keras",
        "config_inferencia_LSTM.json",
        "imputer_Sentinel_LSTM.joblib",
        "imputer_Static_LSTM.joblib",
        "scaler_ERA5_LSTM.joblib",
        "scaler_Sentinel_LSTM.joblib",
        "scaler_Static_LSTM.joblib"
    ]
})

print("\n" + "=" * 100)
print("7. ARTEFACTOS DE INFERENCIA")
print("=" * 100)

display(artefactos)


# ============================================================
# 8. CONTROL DE REPRODUCIBILIDAD
# ============================================================

reproducibilidad = pd.DataFrame([{

    "caso_control":
        "A Coruña - barley - 2023",

    "rendimiento_observado_kg_ha":
        2060.0,

    "prediccion_original_kg_ha":
        2103.297119,

    "prediccion_recargada_kg_ha":
        2103.297119,

    "diferencia_absoluta":
        0.0,

    "reproducible":
        True
}])

print("\n" + "=" * 100)
print("8. CONTROL DE REPRODUCIBILIDAD")
print("=" * 100)

display(reproducibilidad)


# ============================================================
# 9. DESPLIEGUE
# ============================================================

despliegue = pd.DataFrame([{

    "plataforma":
        "Streamlit Community Cloud",

    "modelo":
        "LSTM_multirrama",

    "estado":
        "Operativo",

    "referencias_por_defecto":
        "2017-2022",

    "inputs":
        "ERA5-Land, Sentinel-2, suelo, estructura agrícola y cultivo"
}])

print("\n" + "=" * 100)
print("9. DESPLIEGUE WEB")
print("=" * 100)

display(despliegue)


# ============================================================
# 10. GUARDAR TABLAS
# ============================================================

tabla_dl_validacion.to_csv(
    f"{RESUMEN_DIR}/resumen_arquitecturas_DL_validacion.csv",
    index=False
)

tabla_test.to_csv(
    f"{RESUMEN_DIR}/resumen_LSTM_test2023.csv",
    index=False
)

tabla_cultivos.to_csv(
    f"{RESUMEN_DIR}/resumen_LSTM_test2023_por_cultivo.csv",
    index=False
)

comparacion_final.to_csv(
    f"{RESUMEN_DIR}/comparacion_final_ML_DL.csv",
    index=False
)

artefactos.to_csv(
    f"{RESUMEN_DIR}/artefactos_inferencia.csv",
    index=False
)

reproducibilidad.to_csv(
    f"{RESUMEN_DIR}/control_reproducibilidad.csv",
    index=False
)

despliegue.to_csv(
    f"{RESUMEN_DIR}/despliegue_streamlit.csv",
    index=False
)


# ============================================================
# 11. CONCLUSIÓN
# ============================================================

print("\n" + "=" * 100)
print("CONCLUSIÓN FINAL")
print("=" * 100)

print("""
Arquitecturas DL comparadas:
- LSTM multirrama
- GRU multirrama
- CNN1D multirrama
- ConvLSTM1D multirrama

Arquitectura seleccionada:
LSTM multirrama

Selección:
Validación temporal 2020-2022.

Entrenamiento final:
2017-2022, n = 572.

Test independiente:
2023, n = 95.

Resultados test 2023:
MAE  = 572.012 kg/ha
RMSE = 714.903 kg/ha
R²   = 0.527

El test 2023 no se utilizó para seleccionar la arquitectura,
ajustar hiperparámetros ni reentrenar el modelo.

La inferencia serializada fue reproducible con diferencia
absoluta igual a cero.

El modelo final LSTM multirrama está desplegado mediante
una aplicación web Streamlit.
""")

print("\nArchivos guardados en:")
print(RESUMEN_DIR)

print("\nPASO 15 FINALIZADO")